# Study and transform artist follower counts

We study the scale and skewness of `artist_followers` and add `artist_followers_log1p = ln(1 + artist_followers)` to both datasets. We retain the original column so the representations can be compared during validation.

The prepared copies remain in memory, and we record the findings in `reports/artist_followers_preparation/summary.md`. Raw files and existing processed CSV files remain unchanged when this working notebook runs independently.

## 1. Load the original datasets

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = next(
    (directory for directory in (
        Path.cwd() / "data/raw",
        Path.cwd().parent / "data/raw",
        Path.cwd(),
    ) if all((directory / name).is_file() for name in ("train.csv", "test.csv"))),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Set DATA_DIR to the folder containing train.csv and test.csv.")
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")


## 2. Study follower counts

We inspect missing, negative, and zero counts, then compare quantiles and skewness before and after the logarithm. We describe the transformation separately from its still-unmeasured effect on prediction quality.

In [2]:
def summarize_followers(datasets):
    validity = []
    distributions = []
    for name, frame in datasets.items():
        values = frame["artist_followers"]
        validity.append({
            "dataset": name,
            "missing": int(values.isna().sum()),
            "negative": int(values.lt(0).sum()),
            "zero": int(values.eq(0).sum()),
        })
        if values.lt(0).any():
            raise ValueError("Follower counts must be non-negative before applying log1p.")
        for representation, series in (("original", values), ("log1p", np.log1p(values.astype("float64")))):
            distributions.append({
                "dataset": name,
                "representation": representation,
                "minimum": series.min(),
                "median": series.median(),
                "99th percentile": series.quantile(0.99),
                "maximum": series.max(),
                "skewness": series.skew(),
            })
    return pd.DataFrame(validity), pd.DataFrame(distributions)


def add_log_followers(frame):
    values = frame["artist_followers"]
    if values.lt(0).any():
        raise ValueError("Follower counts must be non-negative before applying log1p.")
    prepared = frame.copy()
    position = prepared.columns.get_loc("artist_followers") + 1
    prepared.insert(position, "artist_followers_log1p", np.log1p(values.astype("float64")))
    return prepared

follower_validity, follower_distributions = summarize_followers({"train": train_raw, "test": test_raw})
display(follower_validity)
display(follower_distributions.round(4))


,dataset,missing,negative,zero
0,train,0,0,2468
1,test,0,0,612


,dataset,representation,minimum,median,99th percentile,maximum,skewness
0,train,original,0.0,6418.0000,5.915204e+07,1.159989e+08,6.9070
1,train,log1p,0.0,8.7670,1.789560e+01,1.856910e+01,0.1592
2,test,original,0.0,6391.0000,7.172041e+07,1.159989e+08,6.8064
3,test,log1p,0.0,8.7628,1.808830e+01,1.856910e+01,0.1655


## 3. Add the logarithmic feature

We apply the natural logarithm of one plus the follower count. This compresses the scale of large counts without discarding popular artists. Zero remains zero, and missing values remain missing. We preserve the original counts, every row, and all other columns.

We will compare the original and logarithmic representations using validation ROC AUC; we have not yet established that adding the transformed feature improves a model.

In [3]:
train_clean = add_log_followers(train_raw)
test_clean = add_log_followers(test_raw)
for original, cleaned in ((train_raw, train_clean), (test_raw, test_clean)):
    pd.testing.assert_frame_equal(original, cleaned.drop(columns="artist_followers_log1p"))
    np.testing.assert_allclose(
        cleaned["artist_followers_log1p"].to_numpy(),
        np.log1p(original["artist_followers"].to_numpy(dtype=float)),
        equal_nan=True,
    )
    assert cleaned.loc[original["artist_followers"].eq(0), "artist_followers_log1p"].eq(0).all()
    assert cleaned["artist_followers_log1p"].isna().equals(original["artist_followers"].isna())
display(train_clean[["id", "artist_followers", "artist_followers_log1p"]].head())


,id,artist_followers,artist_followers_log1p
0,1,1404,7.247793
1,2,1404,7.247793
2,3,1404,7.247793
3,4,1404,7.247793
4,5,1404,7.247793


## 4. Record our initial decision

In [4]:
report_dir = DATA_DIR.parent.parent / "reports/artist_followers_preparation"
report_dir.mkdir(parents=True, exist_ok=True)
lines = [
    "# Artist follower transformation", "",
    "We examined the distribution and validity of `artist_followers` in the original training and test datasets.", "",
    "| Dataset | Missing | Negative | Zero |", "| --- | ---: | ---: | ---: |",
]
for _, row in follower_validity.iterrows():
    lines.append(f"| {row['dataset']} | {row['missing']} | {row['negative']} | {row['zero']} |")
lines += [
    "", "| Dataset | Representation | Minimum | Median | 99th percentile | Maximum | Skewness |",
    "| --- | --- | ---: | ---: | ---: | ---: | ---: |",
]
for _, row in follower_distributions.iterrows():
    lines.append(
        f"| {row['dataset']} | {row['representation']} | {row['minimum']:.4f} | {row['median']:.4f} | "
        f"{row['99th percentile']:.4f} | {row['maximum']:.4f} | {row['skewness']:.4f} |"
    )
lines += [
    "", "## Initial decision", "",
    "We added `artist_followers_log1p = ln(1 + artist_followers)` to both datasets. "
    "The logarithm compresses large follower counts while preserving their order, and the added 1 allows zero followers to map to zero. "
    "", "",
    "We retained the original `artist_followers` column so we can compare the original and transformed representations during model evaluation. "
    "We did not remove popular artists, change follower counts, or fill missing values as part of this transformation. "
    "The formula uses no target information and requires no statistic fitted on training or test data. "
    "Its effect on model performance remains to be assessed using validation ROC AUC.", "",
    "We included the same transformation in `delivery/music_hit_prediction.ipynb`. "
    "The working notebook keeps its copies in memory and leaves raw and existing processed CSV files unchanged. "
    "The delivery notebook can export the combined preparation to `data/processed/`.", "",
]
(report_dir / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Report saved to {report_dir / 'summary.md'}")


Report saved to /home/goncalotorcato/ist1109358/CD/Projeto/CD/reports/artist_followers_preparation/summary.md
